# 04 · 检索与 RAG（Retrieval-Augmented Generation）

**RAG** 让模型基于「你提供的私有文档」回答，而不是只靠训练时记住的知识。本章基于 `data/docs/sample.txt` 构建本地向量库并做检索增强问答。

运行前请确保：
- 已激活 `.venv` 虚拟环境
- 已配置 `.env` 中的 `OPENAI_API_KEY`

> 详细概念讲解见 `docs/04-retrieval-and-rag.md`。


## 0. 环境检查


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.getenv('OPENAI_API_KEY'):
    raise SystemExit('✗ 未找到 OPENAI_API_KEY，请复制 .env.example 为 .env 并填写。')
print('✓ 环境就绪，API Key 已加载')


## 1. 加载并切分文档

长文档需要切成合适大小的块（chunk），兼顾语义完整与模型窗口。`RecursiveCharacterTextSplitter` 是常用的切分器。


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

DOC_PATH = 'data/docs/sample.txt'
PERSIST_DIR = 'data/chroma'

loader = TextLoader(DOC_PATH, encoding='utf-8')
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40)
splits = splitter.split_documents(docs)
print(f'切分得到 {len(splits)} 个文本块')


## 2. 向量化并存入 Chroma

用 `OpenAIEmbeddings` 把每个块变成向量，存入 `Chroma` 并持久化到磁盘（首次运行会创建，之后可直接加载）。


In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings(model=os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small'))
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings, persist_directory=PERSIST_DIR)
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})


## 3. 构造 RAG 链

检索到的上下文 + 问题 → 模型生成。提示词明确要求「只根据上下文回答，不知道就说不知道」，以减少幻觉。


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'))
prompt = ChatPromptTemplate.from_template(
    '只根据下面的上下文回答，若上下文没有答案就说“我不知道”。\n\n'
    '上下文：\n{context}\n\n问题：{question}'
)

def format_docs(docs):
    return '\n\n'.join(d.page_content for d in docs)

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


## 4. 提问并核对检索到的内容


In [ ]:
question = 'LangChain 的核心抽象有哪些？'
print('问题：', question)
print('回答：', rag_chain.invoke(question))

# 练习时打印实际检索到的块做核对，避免盲目信任模型“编造”的引用
for i, d in enumerate(retriever.invoke(question), 1):
    print(f'\n--- 检索块 {i} ---\n' + d.page_content)


## 5. 下一步

- 智能代理：运行 `examples/05_agents.py`，参见 `docs/05-agents.md`

**常见坑**：没做持久化会导致每次重建向量库，浪费 token；chunk 太大/太小会让检索不准；应打印实际检索块核对来源。
